# Prepare the Rāmāyaṇa full checkpoint for GitHub training

Choose **Runtime → Run all** and approve Drive access. No GPU is needed. This verifies the latest Rāmāyaṇa checkpoint and makes two separate transfer parts, each smaller than 100 MB. It never changes or deletes the original checkpoint. When it says Ready, tell your assistant. The parts and manifest use extra Drive space.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, shutil, tempfile

drive.mount('/content/drive')
source_dir = Path('/content/drive/MyDrive/Sanskrit-LLM/ramayana-v1')
target = Path('/content/drive/MyDrive/Sanskrit-LLM/ramayana-github-transfer')
files = sorted(p for p in source_dir.glob('tokens-*.pt') if Path(str(p) + '.sha256').is_file())
if not files:
    raise RuntimeError('No complete Ramayana checkpoint found in ' + str(source_dir))
source = files[-1]
expected = Path(str(source) + '.sha256').read_text().split()[0]

def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

if sha256(source) != expected:
    raise RuntimeError('Checkpoint checksum mismatch. No files were changed.')
target.mkdir(parents=True, exist_ok=True)
size = source.stat().st_size
part_size = 64 * 1024 * 1024
parts = []
with source.open('rb') as reader:
    index = 0
    while True:
        data = reader.read(part_size)
        if not data:
            break
        name = source.name + f'.part-{index:03d}'
        destination = target / name
        digest = hashlib.sha256(data).hexdigest()
        if destination.exists():
            if destination.stat().st_size != len(data) or sha256(destination) != digest:
                raise RuntimeError(f'Existing transfer part differs: {destination}. Original checkpoint is intact.')
        else:
            tmp = target / (name + '.tmp')
            with tmp.open('wb') as out:
                out.write(data)
                out.flush()
                os.fsync(out.fileno())
            if sha256(tmp) != digest:
                raise RuntimeError('Transfer part verification failed: ' + str(tmp))
            os.replace(tmp, destination)
        parts.append({'name': name, 'bytes': len(data), 'sha256': digest})
        index += 1
if sum(p['bytes'] for p in parts) != size or sha256(source) != expected:
    raise RuntimeError('Source changed while copying. Original checkpoint was not modified.')
manifest = {'source_checkpoint': source.name, 'source_sha256': expected, 'source_bytes': size, 'parts': parts}
manifest_path = target / (source.name + '.transfer.json')
content = json.dumps(manifest, indent=2) + '\n'
if manifest_path.exists() and manifest_path.read_text() != content:
    raise RuntimeError('Existing transfer manifest differs; no existing parts were overwritten.')
if not manifest_path.exists():
    tmp = target / (manifest_path.name + '.tmp')
    tmp.write_text(content)
    os.replace(tmp, manifest_path)
print('Ready:', manifest_path, flush=True)
print(json.dumps(manifest, indent=2), flush=True)
